In [6]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Dense , Embedding , Lambda
from tensorflow import keras
import random
import re

In [ ]:
# Data Collection : 
# This dataset has a header row (Tweet_ID, Username, Text, ...) and only 10,000 rows
df = pd.read_csv("/Users/aayushagarwal/Downloads/LP IV Updated Datasets/CBOW/twitter_dataset.csv" , encoding="latin-1")
tweets = df["Text"].astype(str).sample(min(20000 , len(df)) , random_state=1).tolist()
print(tweets[:3])       

['Factor enjoy property world main despite. Grow wrong sign second color.\nRecord country air mention off call food. Head since author media leg suggest question.\nBecause perform mission physical. Congress bit number. Suggest again beautiful whose agree until morning American.', 'Television sure when others water hold. Value economy from mention activity free simply. Exactly big capital class marriage great give. Family near enjoy for.\nCandidate follow scientist black security police allow. Save middle attorney wide mention hope.', 'In structure there trip professional any him. Offer chair break clear so thank.\nAttention close take enough three small. Dinner base community activity analysis improve know. Her into owner. Paper decision under argue.']


In [8]:
# Tokenization : 
tokenizer = Tokenizer(num_words=3000)   # keep top 3000 words
tokenizer.fit_on_texts(tweets)
word2indx = tokenizer.word_index
idx2word = tokenizer.index_word
vocab_size = 3000
sequences = tokenizer.texts_to_sequences(tweets)
print("Vocab Size : " , vocab_size)

Vocab Size :  3000


In [9]:
# Generate Training Data : 
window_size = 2
contexts , targets = [] , []

for seq in sequences:
    for i in range(window_size , len(seq) - window_size):
        contexts.append(seq[i - 2 : i] + seq[i + 1 : i + 3])
        targets.append(seq[i])

contexts = np.array(contexts)
targets = np.array(targets)
print("Training samples : " , len(contexts))

Training samples :  299853


In [10]:
# Model : 
model = keras.Sequential([
    Embedding(vocab_size , 100),
    Lambda(lambda x : tf.reduce_mean(x , axis=1)),
    Dense(vocab_size , activation="softmax")
])

model.compile(optimizer="adam" , loss="sparse_categorical_crossentropy" , metrics=["accuracy"])

model.fit(contexts , targets , epochs=5 , batch_size=256)

Epoch 1/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.0010 - loss: 7.0062
Epoch 2/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.0012 - loss: 6.8853
Epoch 3/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.0013 - loss: 6.8771
Epoch 4/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.0020 - loss: 6.8626
Epoch 5/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.0033 - loss: 6.8361


In [11]:
# Evaluate 2 : Prediction
n = random.randint(0 , len(contexts) - 1)
context_words = [idx2word[w] for w in contexts[n]]
pred = model.predict(np.array([contexts[n]]))
pred_word = idx2word[np.argmax(pred)]

print(context_words)
print(idx2word[targets[n]])
print(pred_word)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
['thousand', 'nor', 'interview', 'throw']
feeling
try
